# UABR-SLM · Notebook 01 — Wikipedia corpus, tokenizer, exposure-controlled pretraining tokens

**Phase 01 of 11.** CPU only · ~1.5–2.5 h · RAM peak ~10 GB · temp disk ~10 GB · output ~5 GB.

**Inputs:** output of notebook **00 (v2, with `in_main`)**, attached via *Add Input → Your Work*.
**Kaggle settings:** Accelerator **None** · Internet **On** · run with **Save Version → Save & Run All** (background).

**Team decisions implemented here (9 Nov 2026, PLAN §14):**
- Snapshot **`wikimedia/wikipedia` 20231101.en**, pinned revision `b04c8d1…`. One snapshot for pretraining *and* retrieval.
- Corpus = **all PopQA subject + object articles** (forced) + **random articles** until **1.0B natural tokens**.
- **Exposure groups** for every PopQA subject article found, stratified by primary relation × popularity bin, 25 % each:
  Control **+0**, Low **+1**, Medium **+4**, High **+16** *additional* copies of the article's **first 1,024 tokens**.
  Copies are **on top of** the 1B natural tokens and shuffled uniformly through the stream.
- Tokenizer: byte-level BPE, **12k vocab**, 9 special tokens with fixed ids:
  `<PAD>=0 <BOS>=1 <EOS>=2 <USER>=3 <ASSISTANT>=4 <ANSWER>=5 <CALL>=6 <EVIDENCE>=7 </EVIDENCE>=8`.
- Documents are stored as `<BOS> text <EOS>`; no Q&A-formatted text in pretraining.

**Light cleaning:** trailing *References / External links / See also / Further reading / Notes…* sections are cut;
disambiguation pages and non-PopQA articles under 300 characters are dropped.
**Known limitations:** HF Wikipedia text has **no infoboxes**, so some PopQA facts are never stated in prose;
titles are matched exactly (no redirect resolution), so a few PopQA subjects will be missing. Both are measured below.

| output (`/kaggle/working/01_corpus/`) | contents |
|---|---|
| `tokenizer/tokenizer.json` | 12k BPE tokenizer |
| `pretrain/train.bin`, `val.bin` | flat uint16 token streams (`<BOS> doc <EOS>` …) |
| `pretrain/doc_index.parquet` | offset/length/kind (`natural`/`copy`)/title for every document |
| `corpus/articles.parquet` | the natural articles = BM25 retrieval corpus for notebook 02 |
| `exposure/exposure_assignment.csv` | subject article → exposure group, extra copies |
| `exposure/ledger_v0.csv` | per PopQA question: subject found, gold alias in article / in repeated prefix |
| `coverage_*.csv`, `stats.json`, `meta.json`, `gate_report.json` | coverage, counts, provenance, gate |

## 1 · Get the project code

In [ ]:
import os, sys, glob, json, subprocess, platform, time, shutil
ON_KAGGLE = os.path.exists("/kaggle/working")

REPO_URL = "https://github.com/Prajwal-13-dev/uabr_slm.git"
REPO_REF = "main"        # pin to a commit hash for final, reportable runs

def find_code_root():
    if os.environ.get("UABR_CODE"):
        return os.path.abspath(os.environ["UABR_CODE"])
    for base in glob.glob("/kaggle/input/*"):
        for root, dirs, _ in os.walk(base):
            if os.path.isdir(os.path.join(root, "src", "uabr")):
                return root
            if root.count(os.sep) - base.count(os.sep) >= 4:
                dirs[:] = []
    dest = "/tmp/uabr_slm"
    if not os.path.isdir(dest):
        url = REPO_URL
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
            url = url.replace("https://", f"https://{tok}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", url, dest], check=True)
        subprocess.run(["git", "-C", dest, "checkout", "-q", REPO_REF], check=True)
    return dest

CODE_ROOT = find_code_root()
sys.path.insert(0, os.path.join(CODE_ROOT, "src"))
CODE_COMMIT = subprocess.run(["git", "-C", CODE_ROOT, "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip() or "unknown"

from uabr.data import wiki_corpus as W
from uabr.data import tokenizer as T
print("code:", CODE_ROOT, "| commit:", CODE_COMMIT[:10])

## 2 · Configuration

In [ ]:
import numpy as np, pandas as pd
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

OUT = "/kaggle/working/01_corpus" if ON_KAGGLE else os.path.abspath("outputs/01_corpus")
TMP = "/tmp/uabr_wiki"                       # scratch: shards + candidate pool (not saved)
for d in ["tokenizer", "pretrain", "corpus", "exposure"]:
    os.makedirs(os.path.join(OUT, d), exist_ok=True)
os.makedirs(TMP, exist_ok=True)

WIKI_LOCAL_DIR = os.environ.get("WIKI_LOCAL_DIR")   # local testing only; None on Kaggle

cfg = W.CorpusConfig(
    target_natural_tokens=int(float(os.environ.get("TARGET_TOKENS", 1e9))),
    vocab_size=int(os.environ.get("VOCAB", 12000)),
    cap_tokens=1024,
    exposure_groups={"control": 0, "low": 1, "medium": 4, "high": 16},
    candidate_frac=float(os.environ.get("CAND_FRAC", 0.30)),
    tokenizer_sample_chars=int(float(os.environ.get("TOK_SAMPLE_CHARS", 3e8))),
)
T0 = time.time()
def stage(msg): print(f"[{(time.time()-T0)/60:6.1f} min] {msg}", flush=True)
print(f"free disk in /tmp: {shutil.disk_usage('/tmp').free/1e9:.0f} GB")
cfg.to_dict()

## 3 · Load notebook-00 output

In [ ]:
def find_file(name, env):
    if os.environ.get(env):
        return os.environ[env]
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found - attach the notebook-00 output via Add Input → Your Work"
    return hits[0]

POPQA_PATH = find_file("popqa_prepared.jsonl", "POPQA_PREPARED")
popqa = pd.read_json(POPQA_PATH, lines=True)
assert "in_main" in popqa.columns, "notebook-00 output is v1 (no in_main) - re-run notebook 00 first"
SUBJ_TITLES, OBJ_TITLES = W.popqa_titles(popqa)
print(f"PopQA rows {len(popqa):,} | in_main {int(popqa['in_main'].sum()):,} | "
      f"unique subject titles {len(SUBJ_TITLES):,} | unique object titles {len(OBJ_TITLES):,}")

## 4 · Scan the snapshot (41 shards)
Each shard is downloaded, split into **forced** PopQA articles and **fill candidates** (a fixed 30 % hash sample of all other articles), written to `/tmp`, then deleted. Re-running skips finished shards.

In [ ]:
shards = W.list_shards(cfg, WIKI_LOCAL_DIR)
assert shards, "no Wikipedia shards found"
stage(f"{len(shards)} shards")
scan_stats = []
for i, name in enumerate(shards):
    f_out, c_out = f"{TMP}/forced_{i:03d}.parquet", f"{TMP}/cand_{i:03d}.parquet"
    if os.path.exists(c_out):
        scan_stats.append(json.load(open(c_out + ".json"))); continue
    path = W.fetch_shard(name, cfg, f"{TMP}/dl", local=bool(WIKI_LOCAL_DIR))
    forced, cand, st = W.scan_shard(path, SUBJ_TITLES, OBJ_TITLES, cfg)
    forced.to_parquet(f_out, index=False); cand.to_parquet(c_out, index=False)
    json.dump(st, open(c_out + ".json", "w")); scan_stats.append(st)
    if not WIKI_LOCAL_DIR:
        os.remove(path)
    stage(f"shard {i+1}/{len(shards)}: {st['articles']:,} articles, forced {st['forced']}, candidates {st['candidates']:,}")
shutil.rmtree(f"{TMP}/dl", ignore_errors=True)
SNAPSHOT = {k: int(sum(s[k] for s in scan_stats)) for k in scan_stats[0]}
SNAPSHOT

## 5 · Forced PopQA articles and coverage

In [ ]:
forced = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(f"{TMP}/forced_*.parquet"))], ignore_index=True)
forced = (forced.sort_values("n_chars", ascending=False).drop_duplicates("title_n").reset_index(drop=True))
FOUND = set(forced["title_n"])

popqa["subj_title_n"] = popqa["s_wiki_title"].map(W.normalize_title)
popqa["obj_title_n"] = popqa["o_wiki_title"].map(W.normalize_title)
popqa["subj_found"] = popqa["subj_title_n"].isin(FOUND)
popqa["obj_found"] = popqa["obj_title_n"].isin(FOUND)

cov_split = popqa.groupby("split")[["subj_found", "obj_found"]].mean().round(3)
cov_split.loc["in_main(all splits)"] = popqa[popqa["in_main"]][["subj_found", "obj_found"]].mean().round(3)
cov_rel = popqa[popqa["in_main"]].groupby("prop")[["subj_found", "obj_found"]].mean().round(3)
cov_pop = popqa[popqa["in_main"]].groupby("pop_bin")[["subj_found", "obj_found"]].mean().round(3)
cov_split.to_csv(f"{OUT}/coverage_by_split.csv"); cov_rel.to_csv(f"{OUT}/coverage_by_relation.csv")
cov_pop.to_csv(f"{OUT}/coverage_by_popbin.csv")
print(f"forced articles: {len(forced):,} ({forced['n_chars'].sum()/1e6:.0f}M chars)")
print("missing subject titles (examples):", sorted(SUBJ_TITLES - FOUND)[:10])
display(cov_split); display(cov_rel); display(cov_pop)

## 6 · Train the 12k byte-level BPE tokenizer

In [ ]:
cand_meta = pd.concat([pd.read_parquet(f, columns=["id", "u", "n_chars", "lm_val"]).assign(file=f)
                       for f in sorted(glob.glob(f"{TMP}/cand_*.parquet"))], ignore_index=True)
pool_chars = cand_meta["n_chars"].sum()
u_sample = min(1.0, cfg.tokenizer_sample_chars / pool_chars) * cfg.candidate_frac   # u is uniform on [0, candidate_frac)
u_ratio_hi = min(cfg.candidate_frac, u_sample + 0.05 * cfg.candidate_frac)

def iter_candidate_texts(lo, hi):
    for f in sorted(glob.glob(f"{TMP}/cand_*.parquet")):
        d = pd.read_parquet(f, columns=["u", "text"])
        yield from d.loc[(d["u"] >= lo) & (d["u"] < hi), "text"]

def tok_training_texts():
    yield from forced["text"]
    yield from iter_candidate_texts(0.0, u_sample)

stage(f"training tokenizer on ~{(cfg.tokenizer_sample_chars + forced['n_chars'].sum())/1e6:.0f}M chars")
tok = T.train_bpe(tok_training_texts(), vocab_size=cfg.vocab_size)
TOK_PATH = f"{OUT}/tokenizer/tokenizer.json"
tok.save(TOK_PATH)

# chars-per-token on held-out candidate text (not used for training)
ratio_texts, n = [], 0
for t in iter_candidate_texts(u_sample, u_ratio_hi):
    ratio_texts.append(t); n += len(t)
    if n > 20_000_000: break
CHARS_PER_TOKEN = n / sum(len(a) for a in T.encode_batch(tok, ratio_texts))
stage(f"vocab {tok.get_vocab_size()} | chars/token on held-out text = {CHARS_PER_TOKEN:.3f}")
print(tok.encode("George Rankin was an Australian politician.").tokens)

## 7 · Tokenize forced articles; choose random fill to reach 1.0B natural tokens

In [ ]:
forced_ids = []
for s in range(0, len(forced), 2000):
    forced_ids += T.encode_batch(tok, forced["text"].iloc[s:s+2000].tolist())
forced["n_tokens"] = [len(a) + 2 for a in forced_ids]          # + <BOS>/<EOS>
FORCED_TOKENS = int(forced["n_tokens"].sum())

fill_budget_tokens = cfg.target_natural_tokens - FORCED_TOKENS
assert fill_budget_tokens > 0
# +2 per doc for <BOS>/<EOS>, approximated with the mean doc length of the pool
mean_doc_chars = cand_meta["n_chars"].mean()
char_budget = fill_budget_tokens * CHARS_PER_TOKEN * (1 - 2 / (mean_doc_chars / CHARS_PER_TOKEN + 2))
fill_meta = W.select_fill(cand_meta, char_budget)
u_max = fill_meta["u"].max()
val_meta = cand_meta[cand_meta["lm_val"] & (cand_meta["u"] <= u_max)]
stage(f"forced tokens {FORCED_TOKENS/1e6:.1f}M | fill articles {len(fill_meta):,} | LM-val articles {len(val_meta):,}")

def load_texts(meta):
    out = []
    for f, ids in meta.groupby("file")["id"]:
        d = pd.read_parquet(f, columns=["id", "title", "text", "n_chars"])
        out.append(d[d["id"].isin(set(ids))])
    return pd.concat(out, ignore_index=True)

fill = load_texts(fill_meta); val_docs = load_texts(val_meta)
stage(f"loaded fill text {fill['n_chars'].sum()/1e9:.2f}B chars, val text {val_docs['n_chars'].sum()/1e6:.1f}M chars")

## 8 · Exposure groups (Control +0 / Low +1 / Medium +4 / High +16)

In [ ]:
SUBJ_FOUND = FOUND & SUBJ_TITLES
exposure = W.assign_exposure(popqa, SUBJ_FOUND, cfg)
title_to_forced = dict(zip(forced["title_n"], range(len(forced))))
capped = {t: forced_ids[title_to_forced[t]][: cfg.cap_tokens] for t in exposure["title_n"]}
exposure["copy_tokens"] = exposure["title_n"].map(lambda t: len(capped[t]) + 2)
exposure["article_tokens"] = exposure["title_n"].map(lambda t: int(forced["n_tokens"].iloc[title_to_forced[t]]))
COPY_TOKENS = int((exposure["copy_tokens"] * exposure["extra_copies"]).sum())

bal = pd.crosstab(exposure["primary_prop"], exposure["exposure_group"], normalize="index").round(3)
print(exposure["exposure_group"].value_counts().to_string())
print(f"extra copy tokens: {COPY_TOKENS/1e6:.1f}M ({COPY_TOKENS/cfg.target_natural_tokens:.1%} on top of natural)")
exposure.to_csv(f"{OUT}/exposure/exposure_assignment.csv", index=False)
display(bal)
display(pd.crosstab(exposure["pop_bin"], exposure["exposure_group"]))

## 9 · Write the shuffled pretraining stream
Natural documents (forced + fill) and all exposure copies are put in one random order (fixed seed) and written as uint16 to `train.bin`. LM-validation articles go to `val.bin`. This is the longest step.

In [ ]:
# order entries: ("f", i) forced article i | ("r", j) fill article j | ("c", title) one copy of a subject prefix
order = [("f", i) for i in range(len(forced))] + [("r", j) for j in range(len(fill))]
order += [("c", t) for t, k in zip(exposure["title_n"], exposure["extra_copies"]) for _ in range(k)]
perm = np.random.default_rng(cfg.seed_shuffle).permutation(len(order))
stage(f"{len(order):,} documents to write")

w = W.TokenWriter(f"{OUT}/pretrain/train.bin")
CHUNK = 4096
for s in range(0, len(perm), CHUNK):
    block = [order[k] for k in perm[s:s+CHUNK]]
    need = [j for kind, j in block if kind == "r"]
    enc = dict(zip(need, T.encode_batch(tok, fill["text"].iloc[need].tolist()))) if need else {}
    for kind, ref in block:
        if kind == "f":
            w.write(T.wrap_doc(forced_ids[ref]), "natural", forced["title_n"].iloc[ref])
        elif kind == "r":
            w.write(T.wrap_doc(enc[ref]), "natural", fill["title"].iloc[ref])
        else:
            w.write(T.wrap_doc(capped[ref]), "copy", ref)
    if (s // CHUNK) % 100 == 0:
        stage(f"  {s+len(block):,}/{len(perm):,} docs, {w.n/1e6:,.0f}M tokens")
doc_index = w.close()
doc_index.to_parquet(f"{OUT}/pretrain/doc_index.parquet", index=False)

wv = W.TokenWriter(f"{OUT}/pretrain/val.bin")
for s in range(0, len(val_docs), CHUNK):
    for a, t in zip(T.encode_batch(tok, val_docs["text"].iloc[s:s+CHUNK].tolist()), val_docs["title"].iloc[s:s+CHUNK]):
        wv.write(T.wrap_doc(a), "natural", t)
val_index = wv.close()

NATURAL_TOKENS = int(doc_index.loc[doc_index["kind"] == "natural", "length"].sum())
WRITTEN_COPY_TOKENS = int(doc_index.loc[doc_index["kind"] == "copy", "length"].sum())
stage(f"train.bin: {w.n/1e9:.3f}B tokens (natural {NATURAL_TOKENS/1e9:.3f}B + copies {WRITTEN_COPY_TOKENS/1e6:.1f}M) | val.bin: {wv.n/1e6:.1f}M tokens")

## 10 · Exposure ledger v0 (own article only, observational)

In [ ]:
articles = dict(zip(forced["title_n"], forced["text"]))
capped_text = {t: tok.decode(a.tolist()) for t, a in capped.items()}
ledger = W.ledger_v0(popqa, articles, capped_text, exposure)
ledger = ledger.merge(popqa[["id", "split", "prop", "pop_bin", "in_main"]], on="id")
ledger.to_csv(f"{OUT}/exposure/ledger_v0.csv", index=False)

m = ledger[ledger["in_main"] & ledger["subj_found"]]
print("gold alias present in subject's own article / in repeated prefix (in_main, found):")
display(m.groupby("exposure_group")[["obj_in_article", "obj_in_capped", "est_fact_exposures"]].mean().round(3))
display(m.groupby("prop")[["obj_in_article", "obj_in_capped"]].mean().round(3).sort_values("obj_in_article"))

## 11 · Save retrieval corpus + metadata

In [ ]:
corpus = pd.concat([
    forced[["id", "title", "text"]].assign(source="popqa_forced", is_subj=forced["is_subj"], is_obj=forced["is_obj"]),
    fill[["id", "title", "text"]].assign(source="random_fill", is_subj=False, is_obj=False),
], ignore_index=True)
corpus.to_parquet(f"{OUT}/corpus/articles.parquet", index=False, compression="zstd")

stats = {
    "snapshot": SNAPSHOT, "chars_per_token": CHARS_PER_TOKEN,
    "forced_articles": len(forced), "fill_articles": len(fill), "lm_val_articles": len(val_docs),
    "natural_tokens": NATURAL_TOKENS, "copy_tokens": WRITTEN_COPY_TOKENS, "train_tokens": int(w.n),
    "val_tokens": int(wv.n), "n_train_docs": len(doc_index),
    "exposure_counts": exposure["exposure_group"].value_counts().to_dict(),
    "subject_coverage_in_main": float(popqa.loc[popqa["in_main"], "subj_found"].mean()),
    "object_coverage_in_main": float(popqa.loc[popqa["in_main"], "obj_found"].mean()),
}
json.dump(stats, open(f"{OUT}/stats.json", "w"), indent=2)
json.dump({"notebook": "01_corpus_tokenizer", "config": cfg.to_dict(),
           "popqa_input": POPQA_PATH, "tokenizer": {"specials": T.SPECIAL_TOKENS, "vocab": tok.get_vocab_size()},
           "bin_format": {"dtype": "uint16", "doc": "<BOS> tokens <EOS>"},
           "code": {"repo": REPO_URL, "ref": REPO_REF, "commit": CODE_COMMIT},
           "env": {"python": platform.python_version()}},
          open(f"{OUT}/meta.json", "w"), indent=2, default=str)
for f in sorted(glob.glob(f"{OUT}/**/*", recursive=True)):
    if os.path.isfile(f):
        print(f"{os.path.getsize(f)/1e6:10.1f} MB  {os.path.relpath(f, OUT)}")
stage("saved")

## 12 · Gate 01

In [ ]:
checks = []
def add(name, ok, detail, severity="fail"):
    checks.append({"name": name, "passed": bool(ok), "detail": detail, "severity": severity})

dev = NATURAL_TOKENS / cfg.target_natural_tokens - 1
add("natural_tokens_within_2pct", abs(dev) <= 0.02, f"{NATURAL_TOKENS:,} ({dev:+.2%} vs target)")
add("copy_tokens_match_plan", WRITTEN_COPY_TOKENS == COPY_TOKENS, f"{WRITTEN_COPY_TOKENS:,} written / {COPY_TOKENS:,} planned")
add("vocab_size", tok.get_vocab_size() == cfg.vocab_size, f"{tok.get_vocab_size()}")
T.check_special_ids(tok); add("special_token_ids", True, "ids 0..8 fixed")
train_mm = np.memmap(f"{OUT}/pretrain/train.bin", dtype=np.uint16, mode="r")
add("max_token_id_in_vocab", int(train_mm[: 50_000_000].max()) < tok.get_vocab_size(), "first 50M tokens checked")
add("bin_length_matches_index", len(train_mm) == int(doc_index["length"].sum()), f"{len(train_mm):,} tokens")
rng = np.random.default_rng(0)
sample = fill["text"].sample(min(200, len(fill)), random_state=0).tolist()
add("tokenizer_roundtrip_lossless", all(tok.decode(a.tolist()) == t for a, t in zip(T.encode_batch(tok, sample), sample)),
    "200 random fill articles decode exactly")
add("val_tokens_min_1M", wv.n >= 1_000_000, f"{wv.n:,} val tokens", "warn")
add("no_popqa_article_in_val", not set(val_docs["title"].map(W.normalize_title)) & FOUND, "LM-val disjoint from forced articles")
add("val_disjoint_from_train", not set(val_docs["id"]) & set(fill["id"]), "")
cov = stats["subject_coverage_in_main"]
add("subject_coverage_ge_85pct", cov >= 0.85, f"{cov:.1%} of in_main questions have their subject article", "warn")
shares = exposure["exposure_group"].value_counts(normalize=True)
add("exposure_groups_balanced_1pp", (shares - 0.25).abs().max() <= 0.01, shares.round(4).to_dict())
add("exposure_balanced_per_relation_3pp", float((bal - 0.25).abs().max().max()) <= 0.03 if len(bal) else True,
    f"max deviation {float((bal - 0.25).abs().max().max()):.3f} (small relations are noisy)", "warn")

fails = [c for c in checks if not c["passed"] and c["severity"] == "fail"]
warns = [c for c in checks if not c["passed"] and c["severity"] == "warn"]
status = "FAIL" if fails else ("PASS_WITH_WARNINGS" if warns else "PASS")
for c in checks:
    print(f"[{'PASS' if c['passed'] else c['severity'].upper():4}] {c['name']:<36} {c['detail']}")
json.dump({"gate": "01_corpus", "status": status, "checks": checks}, open(f"{OUT}/gate_report.json", "w"), indent=2, default=str)
print("\nGATE 01:", status, f"| total time {(time.time()-T0)/60:.0f} min")
assert not fails, "Corpus gate failed - fix before pretraining."

## Next
- **Notebook 02** (BM25 windows) reads `corpus/articles.parquet` + notebook-00 output.
- **Notebook 03** (pretraining) reads `pretrain/train.bin`, `val.bin` and `tokenizer/tokenizer.json`.
- Send the gate output, coverage tables and the ledger table to the team before pretraining: if the gold answer is rarely in the subject's own article, G1 risk is high.